# SO-101 trên Google Colab: ghi demo, train ACT / SmolVLA, DAgger, chấm, đẩy lên Hugging Face

Chạy ô 1-3 mỗi lần mở (hoặc sau khi Colab ngắt), rồi chạy mục cần làm. Mỗi mục chạy **nền** (đóng tab vẫn chạy), lưu mọi thứ lên
Drive `MyDrive/vla/` theo từng phần, nên bị ngắt thì chạy lại ô 1-3 và đúng ô đó: phần đã xong được bỏ qua, train chạy tiếp từ checkpoint.

| Mục | Làm gì | Thời gian (L4) |
|---|---|---|
| 4 | Ghi demo bằng policy viết tay (thầy giáo) | 500 ván ≈12 phút |
| 5 | Train ACT từ đầu + chấm mọi checkpoint | 60k step ≈4 giờ + chấm |
| 6 | Một vòng DAgger: AI lái, thầy sửa, train tiếp, chấm | ≈4-5 giờ |
| 7 | Chấm một mạng / gộp nhiều checkpoint trên 774 seed | ≈40 phút một mạng |
| 8 | Fine-tune SmolVLA + chấm | 20k step ≈6 giờ + chấm |
| 8b | Train Diffusion Policy + chấm | 30k step ≈3-4 giờ + chấm |
| 9 | Đẩy dataset lên Hugging Face | vài phút |
| 10 | Xem tiến độ mọi việc đang chạy | |
| 12 | **Camera rear (từ 2026-11-27):** ghi demo, train song song ACT + SmolVLA, DAgger | xem mục 12 |

Kết quả tới giờ và lý do từng lựa chọn: `docs/so_tay_ai.md`. Tốt nhất: ACT gộp 3 checkpoint DAgger, **84,8 %** trên 774 seed.

Từ 2026-11-27 bộ camera chính là **rear** (sát sau đế tay, giống bộ thật) + cổ tay: làm ở mục 12. Mục 4-8b là chuỗi camera
trên cao cũ, giữ để dựng lại kết quả.

**Trước khi chạy:** *Runtime -> Change runtime type -> GPU* (**L4** hoặc A100).

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Nối Google Drive

Dataset, checkpoint, kết quả chấm đều nằm trong `MyDrive/vla` (giống `D:/vla` ở máy nhà). Colab hỏi quyền Drive: bấm cho phép.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
VLA = '/content/drive/MyDrive/vla'          # thư mục gốc trên Drive
for sub in ('datasets', 'outputs/train', 'eval', 'replays'):
    os.makedirs(f'{VLA}/{sub}', exist_ok=True)
print('Drive sẵn sàng:', VLA)

## 3. Lấy code, cài thư viện, hàm dùng chung

Code lấy từ GitHub (nhánh `my-work`), thư mục phải tên `sim_engine` (tên gói Python). `MUJOCO_GL=egl`: Colab không có màn hình,
MuJoCo vẽ ảnh camera bằng GPU. Cài ≈3-5 phút; Colab đòi *Restart session* thì restart rồi chạy lại từ ô 2.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['HF_HOME'] = '/content/hf-cache'
# Khai báo driver EGL của NVIDIA: thiếu file này MuJoCo vẽ ảnh bằng CPU, chậm ~160 lần
p = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(p):
    open(p, 'w').write('{"file_format_version" : "1.0.0", "ICD" : {"library_path" : "libEGL_nvidia.so.0"}}')
if os.path.isdir('/content/sim_engine/.git'):
    !git -C /content/sim_engine pull -q
else:
    !git clone -q -b my-work https://github.com/tvhhai/sim-engine.git /content/sim_engine
!git -C /content/sim_engine log --oneline -1
# smolvla, diffusion: thư viện phụ cho mục 8 / 8b (transformers, accelerate, diffusers); ACT không cần nhưng cài chung cho gọn
!pip install -q "lerobot[dataset,training,smolvla,diffusion]==0.7.0" "mujoco>=4.0.1" gymnasium opencv-python-headless h5py

In [ ]:
# Hàm dùng chung: mỗi mục ghép các dòng lệnh shell rồi chạy nền bằng bg().
import subprocess
D = '/content/datasets'                      # dataset giải nén ra ổ của máy Colab (đọc nhanh hơn từ Drive)
T = f'{VLA}/outputs/train'                   # checkpoint lưu thẳng lên Drive

import os, subprocess
def bg(name, lines):
    """Ghi các dòng lệnh vào /content/<name>.sh và chạy nền; log ở /content/<name>.log (mục 10 đọc).
    set -e: lệnh nào lỗi thì dừng cả việc. Kết thúc tạo file đánh dấu /content/<name>.done (thành công) hoặc <name>.failed (lỗi), để
    việc khác chờ đúng (wait_for) thay vì tìm chữ trong log. ALL_DONE / JOB_FAILED cuối log chỉ để người đọc."""
    head = ['set -xe', f'rm -f /content/{name}.done /content/{name}.failed',
            f"trap 'rc=$?; if [ $rc -eq 0 ]; then touch /content/{name}.done; echo ALL_DONE; "
            f"else touch /content/{name}.failed; echo JOB_FAILED rc=$rc; fi' EXIT",
            'export MUJOCO_GL=egl HF_HOME=/content/hf-cache', f'cd /content && mkdir -p {D}']
    open(f'/content/{name}.sh', 'w').write('\n'.join(head + lines) + '\n')
    subprocess.Popen(f'bash /content/{name}.sh > /content/{name}.log 2>&1', shell=True, start_new_session=True)
    print('Đã chạy nền:', name, '| xem tiến độ ở mục 10')

def wait_for(name):
    """Dòng shell: chờ việc nền `name` xong; việc đó lỗi thì dừng luôn việc đang chờ (không chạy tiếp với kết quả thiếu)."""
    return (f'until [ -f /content/{name}.done ]; do [ -f /content/{name}.failed ] && {{ echo "{name} lỗi, dừng"; exit 1; }}; '
            f'sleep 60; done')

def upload(src, zip_path):
    """Dòng shell: zip thư mục src rồi chép lên Drive qua file .tmp, xong mới đổi tên -> ai chờ zip_path không đọc phải file dở."""
    tmp = '/content/' + os.path.basename(zip_path)
    return f'[ -f {zip_path} ] || (cd {src} && rm -f {tmp} && zip -qr {tmp} . && cp {tmp} {zip_path}.tmp && mv {zip_path}.tmp {zip_path})'

def unzip(pattern):
    """Giải nén mọi zip khớp pattern (trên Drive) vào D. Giải nén vào thư mục .tmp rồi mới đổi tên và tạo dấu <tên>.complete:
    lần trước chết giữa chừng thì thư mục dở bị làm lại, không bị coi là xong."""
    return (f'for Z in {VLA}/datasets/{pattern}.zip; do P={D}/$(basename $Z .zip); [ -f $P.complete ] && continue; '
            f'rm -rf $P $P.tmp && mkdir -p $P.tmp && unzip -q $Z -d $P.tmp && mv $P.tmp $P && touch $P.complete || exit 1; done')

def merge(name, sources):
    """Gộp nhiều dataset (đường dẫn / glob trong D) thành D/name. Gộp vào name.tmp, xong mới đổi tên + dấu name.complete."""
    out = f'{D}/{name}'
    return (f'[ -f {out}.complete ] || (rm -rf {out} {out}.tmp && python -m sim_engine.learning.merge_datasets --out {out}.tmp '
            f'--repo-id local/{name} ' + ' '.join(f'{D}/{s}' for s in sources) + f' && mv {out}.tmp {out} && touch {out}.complete)')

def train(cfg, out, ds, extra=''):
    """Train theo config YAML. Đã có checkpoint ĐẦY ĐỦ (pretrained_model + train_config + model + training_state) thì train tiếp từ
    checkpoint mới nhất, không cần lối tắt `last` (Drive có thể mất nó). Có thư mục checkpoints mà không cái nào đầy đủ thì DỪNG
    báo lỗi, không bao giờ xoá; chỉ xoá `out` khi chưa có checkpoint nào."""
    ck = (f'CK=$(for d in $(ls -d {out}/checkpoints/[0-9]*/pretrained_model 2>/dev/null | sort); do '
          f'[ -f $d/train_config.json ] && [ -f $d/model.safetensors ] && [ -d $d/../training_state ] && echo $d; done | tail -n 1)')
    return [ck,
            f'if [ -n "$CK" ]; then echo "Train tiếp từ $CK"; lerobot-train --config_path=$CK/train_config.json --resume=true --dataset.root={D}/{ds}',
            f'elif [ -d {out}/checkpoints ] && [ -n "$(ls -A {out}/checkpoints)" ]; then echo "Có {out}/checkpoints nhưng không có checkpoint đầy đủ: không xoá, dừng"; exit 1',
            f'else rm -rf {out}; python -m sim_engine.learning.train sim_engine/learning/configs/{cfg}.yaml --dataset.repo_id=local/{ds} '
            f'--dataset.root={D}/{ds} --output_dir={out} --job_name=$(basename {out}) {extra}; fi']

def score(out, tag, par=3):
    """Chấm mọi checkpoint của out trên 774 seed (par lượt song song) -> eval/full_<tag>_<step>.json; đã chấm thì bỏ qua.
    Chỉ dùng cho điểm CUỐI; chọn checkpoint dùng quick() trên bộ validation."""
    return [f'ls {out}/checkpoints | grep -E "^[0-9]+$" | while read k; do [ -f {VLA}/eval/full_{tag}_$k.json ] || echo $k; done '
            f'| xargs -r -P {par} -I KK sh -c "python -m sim_engine.learning.eval_policy --policy {out}/checkpoints/KK/pretrained_model '
            f'--name {tag}_KK --out {VLA}/eval/full_{tag}_KK.json > /content/full_{tag}_KK.log 2>&1"',
            f'for f in {VLA}/eval/full_{tag}_*.json; do echo "CK $(basename $f) $(grep -m1 \'"all"\' $f)"; done']

def quick(out, tag, cks, extra='', job=None):
    """Chờ từng checkpoint cks của out lưu xong rồi chấm nhanh 100 seed của bộ VALIDATION (2500-2999)
    -> eval/v100_<tag>_<ck>.json. job = tên việc train: train đã dừng (done / failed) mà chưa có checkpoint thì dừng, không chờ mãi."""
    lines = []
    for k in cks:
        p = f'{out}/checkpoints/{k}/pretrained_model'
        stop = (f'{{ [ -f /content/{job}.done ] || [ -f /content/{job}.failed ]; }} && [ ! -f {p}/model.safetensors ] && '
                f'{{ echo "train {job} đã dừng, không có {k}"; exit 1; }}; ') if job else ''
        lines += [f'while [ ! -f {p}/model.safetensors ]; do {stop}sleep 60; done; sleep 60',
                  f'[ -f {VLA}/eval/v100_{tag}_{k}.json ] || python -m sim_engine.learning.eval_policy --policy {p} --seeds val --limit 100 '
                  f'{extra} --name {tag}_{k} --out {VLA}/eval/v100_{tag}_{k}.json > /content/v_{tag}_{k}.log 2>&1']
    return lines

def best(pattern):
    """Lệnh shell in thư mục checkpoint có điểm cao nhất trong các file eval/<pattern>.json (để DAgger / train tiếp tự chọn)."""
    return (f'python -c "import json,glob; f=max(glob.glob(\'{VLA}/eval/{pattern}.json\'), key=lambda p: '
            f'int(json.load(open(p))[\'summary\'][\'all\'].split(\'/\')[0])); print(json.load(open(f))[\'policy\'])"')

# Kiểm tra nhanh: dựng env, vẽ một ảnh camera
%cd /content
from sim_engine.learning import sim_setup
env, opt = sim_setup.make_env(3000)
obs, _ = env.reset(seed=3000, options=opt)
print({k: getattr(v, 'shape', v) for k, v in sim_setup.to_frame(env, obs, [0] * 7, 'test', ('top', 'wrist')).items()})
env.close()

## 4. Ghi demo (thầy giáo = policy viết tay mặc định, bản min-jerk)

`learning/record_dataset.py`, chỉ giữ ván thành công. Chia `PARTS` phần chạy song song, mỗi phần một dải seed riêng (3000+; bộ chấm
dùng 0-2499, DAgger dùng 6000+), xong phần nào zip lên Drive phần đó, cuối cùng gộp thành `D/NAME`.
Lưu ý: vị trí hộp lặp theo `seed % 200` (200 ô). Bản cũ cho các phần cách nhau 400 seed: mỗi phần 100 ván chỉ phủ ≈100 ô đầu của
dải -> 5 phần trùng nhau, dataset `so101_box4_topwrist_mj` chỉ phủ ≈1/2 số ô. Đã sửa: phần k dùng seed 3000+k, 3000+k+PARTS, ...
(`--seed-step PARTS`): các phần không bao giờ trùng seed (dải liền nhau thì phần có ván hỏng lấn sang dải phần sau) và cùng nhau
phủ đủ 200 ô.
`so101_box4_topwrist_mj` (5 x 100 ván) đã có trên Drive: chạy ô này chỉ giải nén và gộp.

In [ ]:
NAME, PARTS, PER_PART = 'so101_box4_topwrist_mj', 5, 100
part = [f'rec() {{ P={D}/{NAME}_p$1; Z={VLA}/datasets/{NAME}_p$1.zip',
        '  if [ -f $Z ]; then [ -f $P.complete ] || (rm -rf $P && mkdir -p $P && unzip -q $Z -d $P && touch $P.complete); return; fi',
        f'  rm -rf $P; python -m sim_engine.learning.record_dataset --episodes {PER_PART} --cameras top,wrist '
        f'--seed-start $((3000 + $1)) --seed-step {PARTS} --root $P --repo-id local/{NAME}_p$1 > /content/rec_p$1.log 2>&1 || return 1',
        '  (cd $P && zip -qr /content/rec_p$1.zip . && cp /content/rec_p$1.zip $Z.tmp && mv $Z.tmp $Z) && touch $P.complete && echo "Xong ghi phần $1"; }']
bg('record', part + [f'for K in $(seq 0 {PARTS - 1}); do rec $K & done; wait',
                     f'for K in $(seq 0 {PARTS - 1}); do [ -f {VLA}/datasets/{NAME}_p$K.zip ] || exit 1; done',
                     merge(NAME, [f'{NAME}_p[0-9]*'])])

## 5. Train ACT từ đầu + chấm mọi checkpoint

`learning/configs/act_scratch.yaml` (camera trên cao + cổ tay, 60k step). Chấm mọi checkpoint trên 774 seed: điểm giữa các checkpoint
chênh nhau nhiều, checkpoint cuối không phải tốt nhất. Lưu ý (sổ tay mục 13): train từ đầu chỉ ≈59 %, phải làm tiếp DAgger (mục 6).

In [ ]:
DS, OUT, TAG = 'so101_box4_topwrist_mj', f'{T}/act_scratch_mj_colab', 'scratch'
bg('scratch', [unzip(f'{DS}_p[0-9]*'), merge(DS, [f'{DS}_p[0-9]*'])] + train('act_scratch', OUT, DS) + score(OUT, TAG))

## 6. Một vòng DAgger: AI lái, thầy sửa, train tiếp, chấm

`learning/dagger.py`: mỗi ván AI tự lái một đoạn ngẫu nhiên (15-250 bước) rồi thầy tiếp quản và làm nốt, ghi đoạn thầy làm. Ghi
`PARTS` x `CHUNK` ván (3 phần song song), gộp với demo gốc + mọi vòng trước, train tiếp `BASE` bằng `act_finetune.yaml` (lr giảm dần),
chấm mọi checkpoint. Vòng sau: tăng `ROUND`, đổi `DRIVERS` / `BASE` sang mạng tốt nhất vừa có (xem kết quả ở mục 10).

In [ ]:
ROUND = 4
# AI lái khi ghi DAgger: một hoặc nhiều checkpoint (nhiều = gộp, trung bình action). Vòng 4: gộp 3 checkpoint tốt nhất (84,8 %).
DRIVERS = [f'{T}/act_box4_topwrist_500_dagger2_colab/checkpoints/010000/pretrained_model',
           f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/020000/pretrained_model',
           f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/030000/pretrained_model']
BASE = f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/020000/pretrained_model'   # điểm xuất phát để train tiếp
DEMO = 'so101_box4_topwrist_500'      # demo gốc của chuỗi DAgger này (500 ván)
PARTS, CHUNK, STEPS = 10, 50, 20000
DAG = lambda r: f'so101_box4_topwrist_dagger{r}'
MERGED, OUT = f'so101_box4_topwrist_500_dagger{ROUND}', f'{T}/act_box4_topwrist_500_dagger{ROUND}_colab'
SEED0 = 6000 + (ROUND - 1) * 10000    # mỗi vòng một dải seed riêng
# Phần K dùng seed SEED0+K, SEED0+K+PARTS, ... (--seed-step): không trùng giữa các phần, phủ đủ 200 ô (seed % 200). Cách cũ
# (SEED0 + K * 1000, seed liền nhau): 1000 chia hết cho 200 nên mọi phần dùng chung các ô đầu dải.

open('/content/dag_part.sh', 'w').write('\n'.join([
    f'K=$1; P={D}/{DAG(ROUND)}_p$K; Z={VLA}/datasets/{DAG(ROUND)}_p$K.zip',
    'if [ -f $Z ]; then [ -f $P.complete ] || (rm -rf $P && mkdir -p $P && unzip -q $Z -d $P && touch $P.complete); exit 0; fi',
    f'rm -rf $P; python -m sim_engine.learning.dagger --policy {" ".join(DRIVERS)} --episodes {CHUNK} '
    f'--seed-start $(({SEED0} + K)) --seed-step {PARTS} --root $P --repo-id local/{DAG(ROUND)}_p$K > /content/dag_p$K.log 2>&1 || exit 1',
    '(cd $P && zip -qr /content/dag_p$K.zip . && cp /content/dag_p$K.zip $Z.tmp && mv $Z.tmp $Z && rm /content/dag_p$K.zip) || exit 1',
]) + '\n')
bg(f'dagger{ROUND}', [
    unzip(DEMO),
    *[unzip(f'{DAG(r)}_p[0-9]*') for r in range(1, ROUND)],
    f'seq 0 {PARTS - 1} | xargs -P 3 -I KK bash /content/dag_part.sh KK',
    f'for K in $(seq 0 {PARTS - 1}); do [ -f {VLA}/datasets/{DAG(ROUND)}_p$K.zip ] || exit 1; done',
    merge(MERGED, [DEMO] + [f'{DAG(r)}_p[0-9]*' for r in range(1, ROUND + 1)]),
    *train('act_finetune', OUT, MERGED, f'--policy.path={BASE} --steps={STEPS}'),
    *score(OUT, f'd{ROUND}'),
])

## 7. Chấm một mạng, hoặc gộp nhiều checkpoint, trên 774 seed

Nhiều thư mục trong `POLICIES` = gộp: mỗi bước mọi mạng cùng đoán, chạy trung bình action (`eval_policy.act`). Gộp 3 checkpoint
DAgger: 77,8 % -> 84,8 %. Kết quả: `MyDrive/vla/eval/full_<TAG>.json`.

In [ ]:
TAG = 'ens3'
POLICIES = [f'{T}/act_box4_topwrist_500_dagger2_colab/checkpoints/010000/pretrained_model',
            f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/020000/pretrained_model',
            f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/030000/pretrained_model']
bg(f'eval_{TAG}', [f'python -m sim_engine.learning.eval_policy --policy {" ".join(POLICIES)} --name {TAG} '
                   f'--out {VLA}/eval/full_{TAG}.json > /content/full_{TAG}.log 2>&1'])

## 8. Fine-tune SmolVLA + chấm

`learning/configs/smolvla_finetune.yaml`: từ bản học sẵn `lerobot/smolvla_base` (≈450M tham số), khoá phần nhìn, chỉ train phần ra
action, lr giảm dần có sẵn. Camera đổi tên top -> camera1, wrist -> camera2 cho khớp bản học sẵn. Dataset mặc định: đúng dữ liệu của
mạng ACT tốt nhất (demo gốc + DAgger 1-4) để so công bằng với 84,8 %. Chấm chậm hơn ACT (mạng to hơn).

In [ ]:
DS, OUT, TAG = 'so101_box4_topwrist_500_dagger4', f'{T}/smolvla_dagger4_colab', 'smolvla'
SOURCES = ['so101_box4_topwrist_500'] + [f'so101_box4_topwrist_dagger{r}_p[0-9]*' for r in range(1, 5)]
bg('smolvla', [f'mkdir -p {D}/so101_box4_topwrist_500 && [ -f {D}/so101_box4_topwrist_500/meta/info.json ] || '
               f'unzip -q {VLA}/datasets/so101_box4_topwrist_500.zip -d {D}/so101_box4_topwrist_500',
               *[unzip(f'so101_box4_topwrist_dagger{r}_p[0-9]*') for r in range(1, 5)],
               merge(DS, SOURCES)] + train('smolvla_finetune', OUT, DS) + score(OUT, TAG, par=2))

## 8b. Train Diffusion Policy + chấm

`learning/configs/diffusion_scratch.yaml`: sinh action bằng cách khử nhiễu dần, giữ được nhiều cách làm cho cùng một cảnh thay vì
trung bình chúng như ACT. Train từ đầu (ResNet18 lấy trọng số ImageNet), U-Net thu nhỏ (89 triệu tham số), chạy DDIM 10 bước.
Cùng dataset với ACT tốt nhất và SmolVLA. Chạy song song với mục 8 được (L4 24 GB) nhưng cả hai chậm đi.

In [ ]:
DS, OUT, TAG = 'so101_box4_topwrist_500_dagger4', f'{T}/diffusion_dagger4_colab', 'diffusion'
SOURCES = ['so101_box4_topwrist_500'] + [f'so101_box4_topwrist_dagger{r}_p[0-9]*' for r in range(1, 5)]
bg('diffusion', [f'mkdir -p {D}/so101_box4_topwrist_500 && [ -f {D}/so101_box4_topwrist_500/meta/info.json ] || '
                 f'unzip -q {VLA}/datasets/so101_box4_topwrist_500.zip -d {D}/so101_box4_topwrist_500',
                 *[unzip(f'so101_box4_topwrist_dagger{r}_p[0-9]*') for r in range(1, 5)],
                 merge(DS, SOURCES)] + train('diffusion_scratch', OUT, DS) + score(OUT, TAG, par=2))

## 9. Đẩy dataset lên Hugging Face

Chia sẻ cho team, xem từng ván trên web, train ở máy khác bằng `--dataset.repo_id=<HF_USER>/<tên>`. Cần token **Write**: tạo ở
huggingface.co -> Settings -> Access Tokens, rồi thêm vào Colab: biểu tượng chìa khoá (Secrets) bên trái -> `HF_TOKEN`, bật Notebook
access. `PRIVATE = True`: chỉ mình xem được, đổi sang public sau ở trang Settings của repo. Đẩy 500 demo min-jerk + bản con 50 ván.

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
HF_USER, PRIVATE = 'tvhhai', True
NAME = 'so101_box4_topwrist_mj'                        # 500 demo min-jerk (mục 4)
flag = ' --private' if PRIVATE else ''
bg('push', [unzip(f'{NAME}_p[0-9]*'), merge(NAME, [f'{NAME}_p[0-9]*']),
            f'python -m sim_engine.learning.push_dataset --root {D}/{NAME} --repo-id {HF_USER}/so101_box4_mj{flag} || exit 1',
            f'python -m sim_engine.learning.push_dataset --root {D}/{NAME} --repo-id {HF_USER}/so101_box4_mj_50 --first 50{flag} || exit 1'])

## 10. Tiến độ

Mỗi việc nền có log `/content/<tên>.log`; xong thì có file `/content/<tên>.done` và dòng `ALL_DONE`, lỗi thì `<tên>.failed` và
`JOB_FAILED`. Ô dưới in vài dòng cuối có ích của mọi việc: ghi (seed ...),
DAgger (từng phần), train (step: ...), chấm ([k/774] ...), điểm từng checkpoint (CK ...), lỗi.

In [ ]:
!for f in /content/*.log; do echo "== $f"; tr '\r' '\n' < $f | grep -E "^seed [0-9]+|Xong|step:|^CK |\[[0-9]+/774\]|\"all\"|Đã đẩy|ALL_DONE|JOB_FAILED|Error|Traceback" | tail -n 2 | cut -c1-170; done

## 11. Đem checkpoint về máy nhà (khi cần)

Checkpoint nằm trên Drive: `MyDrive/vla/outputs/train/<tên>/checkpoints/<step>/pretrained_model/`. Tải cả thư mục `pretrained_model`
về (vd `D:/vla/outputs/train/...`) rồi chạy `eval_policy --policy <thư mục đó>` như ở nhà.

## 12. Camera rear (từ 2026-11-27): bộ camera giống bộ thật

Camera `rear` (MJCF `rearcam`) gắn sát sau đế tay 12 cm, cao 65 cm trên mặt bàn, chúc xuống bàn (sổ tay, mục camera). Mọi mạng mới
dùng **rear + cổ tay**. Dataset ghi kèm cả camera top cũ (render thêm rẻ, giữ lựa chọn cho sau này), rồi tách bản rear + cổ tay để
train. Thứ tự: 12a ghi demo -> 12b train song song ACT + SmolVLA (mở khoá phần nhìn), chấm nhanh 100 seed từng checkpoint ->
12c DAgger từng vòng (ACT lái, mạng điểm cao nhất tự được chọn). Mỗi ô tự chờ ô trước xong, nên chạy liền cả ba ô cũng được.

### 12a. Ghi 500 ván thầy với camera rear + top + cổ tay, tách bản rear + cổ tay

In [ ]:
NAME, RW, PARTS, PER_PART = 'so101_box4_reartopwrist', 'so101_box4_rearwrist', 10, 50
part = [f'rec() {{ P={D}/{NAME}_p$1; Z={VLA}/datasets/{NAME}_p$1.zip',
        '  if [ -f $Z ]; then [ -f $P.complete ] || (rm -rf $P && mkdir -p $P && unzip -q $Z -d $P && touch $P.complete); return; fi',
        f'  rm -rf $P; python -m sim_engine.learning.record_dataset --episodes {PER_PART} --cameras rear,top,wrist '
        f'--seed-start $((3000 + $1)) --seed-step {PARTS} --root $P --repo-id local/{NAME}_p$1 > /content/rec_p$1.log 2>&1 || return 1',
        '  (cd $P && zip -qr /content/rec_p$1.zip . && cp /content/rec_p$1.zip $Z.tmp && mv $Z.tmp $Z) && touch $P.complete && echo "Xong ghi phần $1"; }']
bg('record_rear', part + [
    f'for K in $(seq 0 {PARTS - 1}); do rec $K & done; wait',
    f'for K in $(seq 0 {PARTS - 1}); do [ -f {VLA}/datasets/{NAME}_p$K.zip ] || exit 1; done',
    merge(NAME, [f'{NAME}_p[0-9]*']),
    f'[ -f {D}/{RW}.complete ] || (rm -rf {D}/{RW} {D}/{RW}.tmp && lerobot-edit-dataset --repo_id local/{NAME} --root {D}/{NAME} '
    f'--new_repo_id local/{RW} --new_root {D}/{RW}.tmp --operation.type remove_feature '
    f'--operation.feature_names "[' + "'observation.images.top'" + ']" && mv {D}/{RW}.tmp {D}/{RW} && touch {D}/{RW}.complete)',
    upload(f'{D}/{RW}', f'{VLA}/datasets/{RW}.zip')])

### 12b. Train song song ACT + SmolVLA (mở khoá phần nhìn), chấm nhanh từng checkpoint

A100 40 GB chạy được cả hai cùng lúc (≈36 GB). ACT: `act_scratch.yaml`, 60k step (≈3 giờ khi chạy chung). SmolVLA:
`smolvla_rear_unfrozen.yaml`, 30k step (≈8 giờ khi chạy chung), chấm với đoán lại sau 10 action. Chờ 12a xong mới bắt đầu.
Chấm nhanh (chọn checkpoint) trên **bộ validation** 2500-2999 (`eval/v100_*.json`), để 774 seed chỉ dùng chấm cuối. Lần chạy đầu
(2026-11-27) còn chấm trên 100 seed của bộ 774 (`eval/q100_*_rw_*.json`).

In [ ]:
RW = 'so101_box4_rearwrist'
ACT_OUT, SV_OUT = f'{T}/act_rearwrist_colab', f'{T}/smolvla_rear_unfrozen_colab'
# Zip trên Drive chỉ xuất hiện khi đã chép xong (upload qua .tmp), nên thấy zip là dùng được, kể cả khi 12a chạy ở runtime trước.
ready = [f'until [ -f {VLA}/datasets/{RW}.zip ]; do [ -f /content/record_rear.failed ] && exit 1; sleep 60; done', unzip(RW)]
bg('act_rw', ready + train('act_scratch', ACT_OUT, RW, '--num_workers=4'))
bg('q_act_rw', quick(ACT_OUT, 'act_rw', ['010000', '020000', '030000', '040000', '050000', '060000'], job='act_rw'))
bg('sv_rw', ready + ['sleep 120'] + train('smolvla_rear_unfrozen', SV_OUT, RW, '--num_workers=6'))
bg('q_sv_rw', quick(SV_OUT, 'sv_rw', ['005000', '010000', '015000', '020000', '025000', '030000'], '--n-action-steps 10', job='sv_rw'))

### 12c. Một vòng DAgger cho chuỗi camera rear

Tăng `ROUND` mỗi vòng. Mạng lái và điểm xuất phát = checkpoint điểm cao nhất trên bộ validation của vòng trước (vòng 1: ACT 12b),
tự chọn khi vòng trước chấm xong (vòng trước lỗi thì vòng này dừng, không chạy với kết quả thiếu). Ghi `PARTS x CHUNK` ván (phần K
dùng seed `SEED0 + K`, bước nhảy `PARTS`), gộp demo + mọi vòng, train tiếp 20k step bằng `act_finetune.yaml` (lr giảm dần), chấm
nhanh từng checkpoint.

In [ ]:
ROUND, PARTS, CHUNK = 1, 10, 30
DEMO = 'so101_box4_rearwrist'
DAG = lambda r: f'so101_box4_rearwrist_dagger{r}'
PREV_TAG = 'act_rw' if ROUND == 1 else f'act_rw_d{ROUND - 1}'
PREV_JOB = 'q_act_rw' if ROUND == 1 else f'dagger{ROUND - 1}_rw'     # việc chấm xong vòng trước
MERGED, OUT = f'so101_box4_rearwrist_dagger{ROUND}m', f'{T}/act_rearwrist_dagger{ROUND}_colab'
SEED0 = 6000 + (ROUND - 1) * 20000
DRV = f'/content/best_{PREV_TAG}.txt'
open(f'/content/dag{ROUND}_part.sh', 'w').write('\n'.join([
    f'K=$1; P={D}/{DAG(ROUND)}_p$K; Z={VLA}/datasets/{DAG(ROUND)}_p$K.zip',
    'if [ -f $Z ]; then [ -f $P.complete ] || (rm -rf $P && mkdir -p $P && unzip -q $Z -d $P && touch $P.complete); exit 0; fi',
    f'rm -rf $P; python -m sim_engine.learning.dagger --policy $(cat {DRV}) --episodes {CHUNK} --seed-start $(({SEED0} + K)) --seed-step {PARTS} '
    f'--root $P --repo-id local/{DAG(ROUND)}_p$K > /content/dag{ROUND}_p$K.log 2>&1 || exit 1',
    f'(cd $P && zip -qr /content/dag{ROUND}_p$K.zip . && cp /content/dag{ROUND}_p$K.zip $Z.tmp && mv $Z.tmp $Z && touch $P.complete) || exit 1']) + '\n')
bg(f'dagger{ROUND}_rw', [
    wait_for(PREV_JOB),
    best(f'v100_{PREV_TAG}_*') + f' > {DRV}', f'cat {DRV}',
    unzip(DEMO),
    *[unzip(f'{DAG(r)}_p[0-9]*') for r in range(1, ROUND)],
    f'seq 0 {PARTS - 1} | xargs -P 4 -I KK bash /content/dag{ROUND}_part.sh KK',
    f'for K in $(seq 0 {PARTS - 1}); do [ -f {VLA}/datasets/{DAG(ROUND)}_p$K.zip ] || exit 1; done',
    merge(MERGED, [DEMO] + [f'{DAG(r)}_p[0-9]*' for r in range(1, ROUND + 1)]),
    *train('act_finetune', OUT, MERGED, f'--policy.path=$(cat {DRV}) --steps=20000'),
    *quick(OUT, f'act_rw_d{ROUND}', ['005000', '010000', '015000', '020000'])])
# Việc dagger{ROUND}_rw tự chấm luôn các checkpoint của nó: vòng sau chờ đúng việc này (PREV_JOB).